In [1]:
import pandas as pd
import numpy as np
import yfinance as yf

# load projected FCF from our projections work
fcf_proj = pd.Series({
    2027: 15841.8,
    2028: 17385.7,
    2029: 18830.0,
    2030: 20184.9,
    2031: 21355.7,
})

# WACC and shares outstanding
wacc = 0.0897
shares_outstanding = yf.Ticker("CRM").info.get('sharesOutstanding') / 1_000_000

print("Projected FCF (millions USD):")
print(fcf_proj)
print(f"\nWACC: {wacc:.2%}")
print(f"Shares Outstanding: {shares_outstanding:,.1f}M")

Projected FCF (millions USD):
2027    15841.8
2028    17385.7
2029    18830.0
2030    20184.9
2031    21355.7
dtype: float64

WACC: 8.97%
Shares Outstanding: 819.0M


In [2]:
# terminal growth rate - conservative, roughly in line with long run GDP
terminal_growth_rate = 0.025

# present value of each projected FCF
# year 1 = 2027, year 2 = 2028, etc.
pv_fcfs = {}
for i, (year, fcf) in enumerate(fcf_proj.items(), start=1):
    pv_fcfs[year] = fcf / (1 + wacc) ** i

pv_fcfs = pd.Series(pv_fcfs).round(1)

# terminal value at end of 2031
terminal_value = (fcf_proj[2031] * (1 + terminal_growth_rate)) / (wacc - terminal_growth_rate)

# discount terminal value back to today (5 years)
pv_terminal_value = terminal_value / (1 + wacc) ** 5

print("Present Value of Projected FCFs:")
print(pv_fcfs)
print(f"\nTerminal Value: ${terminal_value:,.1f}M")
print(f"PV of Terminal Value: ${pv_terminal_value:,.1f}M")
print(f"\nSum of PV FCFs: ${pv_fcfs.sum():,.1f}M")

Present Value of Projected FCFs:
2027    14537.8
2028    14641.3
2029    14552.2
2030    14315.2
2031    13898.9
dtype: float64

Terminal Value: $338,324.5M
PV of Terminal Value: $220,190.5M

Sum of PV FCFs: $71,945.4M


In [3]:
# enterprise value = PV of FCFs + PV of terminal value
enterprise_value = pv_fcfs.sum() + pv_terminal_value

# move from enterprise value to equity value
# equity value = enterprise value - debt + cash
cash = yf.Ticker("CRM").balance_sheet.loc['Cash And Cash Equivalents'].iloc[0] / 1_000_000
total_debt = 17176.0

equity_value = enterprise_value - total_debt + cash

# implied share price
implied_price = equity_value / shares_outstanding
current_price = 183.26

upside_downside = (implied_price / current_price - 1)

print(f"Enterprise Value: ${enterprise_value:,.1f}M")
print(f"Less: Total Debt: ${total_debt:,.1f}M")
print(f"Plus: Cash: ${cash:,.1f}M")
print(f"Equity Value: ${equity_value:,.1f}M")
print(f"\nShares Outstanding: {shares_outstanding:,.1f}M")
print(f"Implied Share Price: ${implied_price:,.2f}")
print(f"Current Market Price: ${current_price:,.2f}")
print(f"Upside / (Downside): {upside_downside:+.1%}")

Enterprise Value: $292,135.9M
Less: Total Debt: $17,176.0M
Plus: Cash: $7,327.0M
Equity Value: $282,286.9M

Shares Outstanding: 819.0M
Implied Share Price: $344.67
Current Market Price: $183.26
Upside / (Downside): +88.1%


In [4]:
# what WACC would justify the current $183 market price?
# solve backwards to find the implied WACC

from scipy.optimize import brentq

def implied_price_at_wacc(w):
    pv = sum(fcf_proj[year] / (1 + w) ** i 
             for i, year in enumerate(fcf_proj.index, start=1))
    tv = (fcf_proj[2031] * (1 + terminal_growth_rate)) / (w - terminal_growth_rate)
    pv_tv = tv / (1 + w) ** 5
    ev = pv + pv_tv
    eq = ev - total_debt + cash
    return eq / shares_outstanding - 183.26

implied_wacc = brentq(implied_price_at_wacc, 0.05, 0.25)
print(f"WACC implied by current $183 price: {implied_wacc:.2%}")
print(f"Our WACC assumption: {wacc:.2%}")
print(f"Difference: {implied_wacc - wacc:.2%}")

WACC implied by current $183 price: 14.14%
Our WACC assumption: 8.97%
Difference: 5.17%
